# 第9回 ペンギンを、楕円の群で分ける（解答例・教員用）


---


## 前回からの接続


第8回は、星をk-meansで分けました。k-meansは、中心までの距離の二乗和を小さくするので、丸い群しか探せず、細長い帯を輪切りにしました。

今日は、群の形が楕円でもよい分け方、混合正規分布を使います。


---


## 今日の分析目標


体の測り方だけで、ペンギンの種は分かれるか。何種類とみなせるかに答える。

まず、くちばしの2本の列で、k-meansと混合正規分布に分け、種と比べます。次に、4本の列で、群の数と群の形（型）のすべての組でBICという物差しを出し、何種類とみなせるかを選びます。そして、1羽ごとに群に入る確率を出し、言い切れない羽を探します。最後に、BICが選んだ成分に名前を付け、使わなかった列で確かめます。

もう1つのねらいは、`for`の中に`for`を入れて、2つの選び方のすべての組を試し、表にためられるようになることです。お手本ごとに、コードを1行ずつ読み（「1行ずつ読む」）、変数の中身を見ます（「中身を見る」）。


### 公開データの世界

南極のパーマー基地の研究者が、近くの3つの島でペンギンを測りました。その測った値が、誰でも使える公開データ（palmerpenguins、CC0）として配られています。統計の教材として、世界中で使われているデータです。


### 場面

南極には、アデリー・ヒゲ・ジェンツーの3種のペンギンがいます。見た目はよく似ていて、体の大きさやくちばしの形が少しずつ違います。研究者は種の名前を記録していますが、測った値だけで見分けられるでしょうか。

種の名前を使わず、体の測り方だけで、ペンギンを群に分けます。分けた群が種と合うか、合うなら、データから見て何種類とみなせるかを確かめます。


### 手元のデータ

1行が1羽で、344羽あります。測った列は4本です：くちばしの長さ（`bill_length_mm`）、くちばしの深さ（`bill_depth_mm`、上下の厚み）、ひれの長さ（`flipper_length_mm`）、体重（`body_mass_g`）。ほかに、種（`species`）・島（`island`）・性別（`sex`）・年（`year`）の列があります。これらは、分けたあとの答え合わせだけに使います。

データはpalmerpenguins（Horst, Hill & Gorman, 2020）として公開されていて、CC0なので自由に使えます。


### 最後に出す答え

体の測り方だけで分けると、3つの種が分かれるか。分かれるなら、データから見て何種類とみなせるかです。


### 今日の一周（結果だけ先に）

コードの説明はあとにして、今日の解析を、結果だけで一度通して見ておきます。

1. 楕円を許す：くちばしの長さと深さを並べると、3種がそれぞれ斜めの楕円にまとまって見えます。楕円を許す混合正規分布のほうが、丸い群のk-meansより、種とよく合います。
2. 群の数を選ぶ：BICで群の数を選ぶと、楕円の型なら3です。えっ、丸の型だと7になります。
3. 境目を読む：混合正規分布は、どの群に入るかを確率で言えます。確率が0.9に届かない羽が、種の境目にいます。
4. 名前を付ける：丸の型の7つの成分は、種と性別の組にかなり重なります。
5. 種の中を分ける：ジェンツーだけを2つに分けると、メスはすべて1つの成分に入ります。

このあと、この5つの段の中身を、1つずつ開けていきます。


### 今日の問いとやること

| 段 | 問い | やること |
|---|---|---|
| 楕円を許す（1節） | 丸い群で足りる？ | くちばし2本でk-meansと混合正規分布に分け、種と比べる |
| 群の数を選ぶ（2節） | 何種類とみなせる？ | 4本の列で、群の数と型のすべての組のBICを、入れ子の`for`で出す |
| 境目を読む（3節） | 言い切れる？ | 所属の確率を出し、0.9に届かない羽を数えて図に印を付ける |
| 名前を付ける（4節） | 7つの成分は何？ | 成分ごとの平均を見て、使わなかった列（島・性別）と突き合わせる |
| 種の中を分ける（5節） | 種の中にも群はある？ | 1つの種だけを2つに分け、性別と突き合わせる |


### 今日の進め方

段ごとに、まずスライドで、その段の結論までを通して聞きます。段の終わりにノートへ移り、その段の節のセルを上から実行して、スライドと同じ数が出るかを確かめます。そして、段ごとに1つあるTODOに答えます。

TODOは、実行の前に結果を予想して書き、1か所を替えて実行し、予想と比べる形です。予想が外れても、かまいません。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 混合正規分布で行を群に分け、k-meansの丸い群との違いを、図と種との食い違いで説明できる
- 群の数と型のすべての組でBICを出し、型ごとに選ばれる群の数を読める
- 入れ子の`for`で、2つの選び方のすべての組を試し、結果を表にためるコードを読める
- 所属の確率を出し、言い切れない点を数えて図に示せる
- 成分ごとの平均で名前の候補を付け、使わなかった列と突き合わせて確かめられる
- 群の中にも群があり、何を群と呼ぶかは問いが決めることを、種とオス・メスの例で説明できる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


---


## 1. 丸い群で足りる？


ここから、段1「楕円を許す」（1節）です。問いは、k-meansの丸い群で、ペンギンの種は分かれるか、です。

まず、表を読み込み、列の意味を読みます。


### お手本：表を読み込む

- 役割：ペンギンの表を読み込み、羽の数を数えて、先頭3行を見る
- 入力：データの置き場所`DATA_DIR`と、ファイル名`penguins.csv`
- 出力：表`raw`（344羽）


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
raw = pd.read_csv(f'{DATA_DIR}/penguins.csv')
print(len(raw), '羽')
raw.head(3)


#### 1行ずつ読む

- `DATA_DIR`の行：データの置き場所を`DATA_DIR`にしまう（Colabでは、置き場所のURLが2行に分かれて入っている）
- `pd.read_csv`の行：置き場所の`penguins.csv`を読み、表`raw`にしまう
- `print`の行と最後の行：羽の数を表示し、先頭3行を見る


### 8本の列

| 列 | 意味 |
|---|---|
| `species`・`island` | 種・島 |
| `bill_length_mm`・`bill_depth_mm` | くちばしの長さ・深さ（mm）。深さは上下の厚み |
| `flipper_length_mm`・`body_mass_g` | ひれ（翼）の長さ（mm）・体重（g） |
| `sex`・`year` | 性別・測った年 |


### お手本：列と値を日本語に

- 役割：列の名前と、表の中の値（種・島・性別）を、日本語にする
- 入力：表`raw`と、2つの辞書`names`・`kana`
- 出力：表`df`


In [ ]:
names = {'species': '種', 'island': '島',
         'bill_length_mm': 'くちばしの長さ',
         'bill_depth_mm': 'くちばしの深さ',
         'flipper_length_mm': 'ひれの長さ',
         'body_mass_g': '体重', 'sex': '性別', 'year': '年'}
kana = {'Adelie': 'アデリー', 'Chinstrap': 'ヒゲ',
        'Gentoo': 'ジェンツー', 'Biscoe': 'ビスコー',
        'Dream': 'ドリーム', 'Torgersen': 'トージャーセン',
        'male': 'オス', 'female': 'メス'}
df = raw.rename(columns=names).replace(kana)
df.head(3)


#### 1行ずつ読む

- 1〜5行目：元の列の名前を鍵、日本語の名前を値にした辞書（第5回）
- 6〜9行目：表の中の値（種・島・性別）を、日本語にする辞書
- 10行目：列の名前を替え（`rename`）、値も替えて（`replace`）、表`df`にしまう
- 11行目：先頭3行を見る


### お手本：欠測の羽を外す

- 役割：分けるのに使う4本の列のどれかが欠測（missing data）の羽を外す
- 入力：表`df`
- 出力：表`df`（342羽）


In [ ]:
cols = ['くちばしの長さ', 'くちばしの深さ', 'ひれの長さ', '体重']
df = df.dropna(subset=cols).reset_index(drop=True)
print(len(df), '羽')


#### 1行ずつ読む

- 1行目：分けるのに使う4本の列の名前を、リスト`cols`にしまう
- 2行目：`cols`のどれかが欠測（NaN）の行を外し（`dropna`）、行の番号を振り直す
- 3行目：残った羽の数を表示する

344羽から342羽になりました。4本の測った値がすべて欠けた羽を外しました。


### 性別の欠測は残す


In [ ]:
print(df['性別'].isna().sum(), '羽')


性別が欠測の羽が、まだ9羽あります。性別は分けるのに使わないので、外しません。答え合わせで性別を数えるときだけ、この9羽は数に入りません。


### お手本：くちばしを描く

- 役割：種ごとに色を変えて、くちばしの長さと深さの点を打つ
- 入力：表`df`の種・くちばしの長さ・くちばしの深さ
- 出力：図


In [ ]:
for s in ['アデリー', 'ヒゲ', 'ジェンツー']:
    d = df[df['種'] == s]
    plt.scatter(d['くちばしの長さ'], d['くちばしの深さ'],
                s=10, label=s)
plt.legend()
plt.xlabel('くちばしの長さ（mm）')
plt.ylabel('くちばしの深さ（mm）')
plt.show()


#### 1行ずつ読む

- 1・2行目：種の名前を1つずつ取り、その種の羽だけの表`d`を作る
- 3・4行目：横に長さ、縦に深さをとって点を打つ。`label`で種の名前を付ける
- 5行目：種の名前の見出し（凡例）を出す
- 6〜8行目：軸の名前を付けて、図を表示する

3種それぞれ、くちばしが長いほど深くなっています。斜めの楕円です。ところが、全体で見ると、長いほど浅い向きに見えます。数で確かめます。


### お手本：相関係数を比べる

- 役割：全体と種ごとに、くちばしの長さと深さの相関係数を出す
- 入力：表`df`
- 出力：4つの相関係数


In [ ]:
print(df['くちばしの長さ'].corr(df['くちばしの深さ']).round(2))
for s in ['アデリー', 'ヒゲ', 'ジェンツー']:
    d = df[df['種'] == s]
    print(s, d['くちばしの長さ'].corr(d['くちばしの深さ']).round(2))


#### 1行ずつ読む

- 1行目：全体の、長さと深さの相関係数（`corr`）を表示する
- 2〜4行目：種ごとに羽を取り出し、同じ相関係数を表示する

全体では負、種の中では正です。群ごとに、傾いた楕円があります。


### お手本：くちばし2本を標準化

第8回と同じく、列の単位をそろえてから分けます。まず、図で見られるように、くちばしの2本だけで分けます。

- 役割：くちばしの2本の列を、平均0・標準偏差（standard deviation）1にそろえる
- 入力：表`df`のくちばしの長さと深さ
- 出力：配列`X2`（羽の数×2）


In [ ]:
from sklearn.preprocessing import StandardScaler
cols2 = ['くちばしの長さ', 'くちばしの深さ']
X2 = StandardScaler().fit_transform(df[cols2])
print(X2[:3].round(2))


#### 1行ずつ読む

- 1行目：標準化（standardization）の道具を呼び出す
- 2・3行目：2本の列の名前を`cols2`にしまい、平均0・標準偏差1にそろえて配列`X2`にする
- 4行目：先頭3羽の値を見る


### お手本：k-meansで3つに

- 役割：くちばし2本で、k-meansで3つに分け、種と組ごとに数える
- 入力：配列`X2`と、群の数3
- 出力：種×群の表


In [ ]:
from sklearn.cluster import KMeans
km = KMeans(3, n_init=10, random_state=0)
km.fit(X2)
pd.crosstab(df['種'], km.labels_)


#### 1行ずつ読む

- 1〜3行目：k-meansを、群の数3で当てはめる（第8回）
- 4行目：種と群の番号の組ごとに、羽を数える（`pd.crosstab`）

群の番号は名前にすぎません。どの群がどの種かは、数を見て読みます。


### お手本：食い違いを数える関数

群ごとに、いちばん多い種を、その群の種とみなします。それ以外の種の羽が、食い違った羽です。同じ数え方を何度も使うので、関数にします（第7回）。

- 役割：群の番号を受け取り、種と食い違う羽の数を返す
- 入力：群の番号`lab`
- 出力：食い違った羽の数


In [ ]:
def n_miss(lab):
    ct = pd.crosstab(df['種'], lab)
    return len(df) - ct.max().sum()

print(n_miss(km.labels_), '羽')


#### 1行ずつ読む

- 1行目：関数`n_miss`を作る。入力は群の番号`lab`
- 2行目：種と群の番号の表を作る
- 3行目：列（群）ごとの最大を足すと、群の種と合う羽の数。全体から引いて返す
- 5行目：k-meansの群で数える

k-meansは、中心からの距離で切るので、斜めの楕円の端が、となりの群に取られます。


In [ ]:
# @title 図：k-meansの3つの群と中心（中心は×。中身は読み飛ばしてよい）
mu = df[cols2].mean().values
sd = df[cols2].std(ddof=0).values
cc = km.cluster_centers_ * sd + mu
plt.scatter(df['くちばしの長さ'], df['くちばしの深さ'], s=10, c=km.labels_)
plt.scatter(cc[:, 0], cc[:, 1], s=250, marker='X', color='black')
plt.title(f'k-means：食い違い {n_miss(km.labels_)}羽')
plt.xlabel('くちばしの長さ（mm）')
plt.ylabel('くちばしの深さ（mm）')
plt.show()


### 混合正規分布の考え

データは、いくつかの正規分布（成分）を重ねたものから出てきた、と考えます。成分ごとに、平均・広がりと傾き（共分散）・割合を持ちます。各点を、出てきたらしい成分に分けます。

$k$個の成分を、割合$\pi_j$で重ねます。

$$
p(x) = \sum_{j=1}^{k} \pi_j \, N(x \mid \mu_j, \Sigma_j)
$$

$\mu_j$は成分$j$の平均、$\Sigma_j$は共分散（広がりと傾き）、$\pi_j$は割合で、足すと1です。


In [ ]:
# @title 図：ひれの長さの分布を、2つの正規分布の重ね合わせで表す（中身は読み飛ばしてよい）
from sklearn.mixture import GaussianMixture
v = df[['ひれの長さ']].values
g = GaussianMixture(2, n_init=5, random_state=0).fit(v)
xs = np.linspace(165, 240, 300)
plt.hist(v[:, 0], bins=30, density=True, color='lightgray')
tot = np.zeros_like(xs)
for j in range(2):
    mu1, s2, w = g.means_[j, 0], g.covariances_[j, 0, 0], g.weights_[j]
    y = w * np.exp(-(xs - mu1) ** 2 / (2 * s2)) / np.sqrt(2 * np.pi * s2)
    tot = tot + y
    plt.plot(xs, y, '--', label=f'成分{j + 1}（割合{w:.2f}）')
plt.plot(xs, tot, color='black', label='重ね合わせ')
plt.xlabel('ひれの長さ（mm）')
plt.ylabel('密度')
plt.legend()
plt.show()


ひれの長さの分布（灰色）を、2つの正規分布の重ね合わせ（黒い線）で表しました。


### お手本：混合正規分布で3つに

- 役割：くちばし2本で、混合正規分布で3つに分け、種と食い違う羽を数える
- 入力：配列`X2`と、成分の数3
- 出力：羽ごとの成分の番号`lab2`と、食い違った羽の数


In [ ]:
from sklearn.mixture import GaussianMixture
gm2 = GaussianMixture(3, n_init=5, random_state=0)
gm2.fit(X2)
lab2 = gm2.predict(X2)
print(n_miss(lab2), '羽')


#### 1行ずつ読む

- 1・2行目：混合正規分布の道具を呼び出し、成分の数3で準備する（出発点は5通り）
- 3・4行目：当てはめ、羽ごとにいちばん出てきたらしい成分の番号を出す
- 5行目：食い違った羽を数える

食い違いは、k-meansの半分ほどです。


#### 中身を見る：割合と平均


In [ ]:
print(gm2.weights_.round(2))
print(gm2.means_.round(2))


割合は、3つの成分にどのくらいの羽が入るかです。平均は標準化した単位です。


In [ ]:
# @title 図：混合正規分布の3つの成分と楕円（中身は読み飛ばしてよい）
from matplotlib.patches import Ellipse
ax = plt.gca()
ax.scatter(df['くちばしの長さ'], df['くちばしの深さ'], s=10, c=lab2)
for j in range(3):
    S = np.diag(sd) @ gm2.covariances_[j] @ np.diag(sd)
    m = gm2.means_[j] * sd + mu
    vals, vecs = np.linalg.eigh(S)
    ang = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    ax.add_patch(Ellipse(m, 4 * np.sqrt(vals[1]), 4 * np.sqrt(vals[0]),
                         angle=ang, fill=False, lw=2))
plt.title(f'混合正規分布：食い違い {n_miss(lab2)}羽')
plt.xlabel('くちばしの長さ（mm）')
plt.ylabel('くちばしの深さ（mm）')
plt.show()


混合正規分布の楕円（平均から標準偏差2つぶん）は、種ごとの斜めの塊をなぞります。


> 重要：k-meansは、どの群も同じ大きさの丸と考えて分ける。混合正規分布は、群ごとに広がりと傾きを持つ楕円を許す。


### 結果を見る前に、予想を書く

群の数を、データに選ばせたら、3を選ぶでしょうか。予想と、その理由を1行で書いてください。


（ここに予想を書く）


### TODO①：列を4本にする（予想 → 実行 → 比べる）

下のセルは、選んだ列を標準化し、k-meansと混合正規分布で3つに分けて、種と食い違う羽の数を2つ並べて表示します。

まず、予想を書いてください。ひれの長さと体重も使うと、食い違いは減るでしょうか。k-meansと混合正規分布のどちらが減るでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`cols2`を`cols`に替えて実行してください。


In [ ]:
# TODO: cols2 を cols に替えて実行する

X_x = StandardScaler().fit_transform(df[cols])
km_x = KMeans(3, n_init=10, random_state=0).fit(X_x)
gm_x = GaussianMixture(3, n_init=5, random_state=0).fit(X_x)
print(n_miss(km_x.labels_), n_miss(gm_x.predict(X_x)))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。食い違いは、どちらで減りましたか。


（ここに答えを書く）


4本にすると、混合正規分布の食い違いは12羽から5羽に減ります。k-meansは25羽から29羽に、かえって増えます。体重のように散らばりの大きい列が入ると、丸い群ではかえって種の塊を切りやすくなります。ここから、4本の列を使います。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 楕円を許す（1節） | 種と食い違う羽：k-means 25、混合正規分布 12（くちばし2本） |
| 群の数を選ぶ（2節） | ？（このあとの段で） |
| 境目を読む（3節） | ？（このあとの段で） |
| 名前を付ける（4節） | ？（このあとの段で） |
| 種の中を分ける（5節） | ？（このあとの段で） |

まだ分からないのは、群の数を3と決めずに、データから選べるか、です。


---


## 2. 何種類とみなせる？


ここから、段2「群の数を選ぶ」（2節）です。問いは、群の数を決めずに、データから何種類とみなせるか、です。

種ごとの斜めの楕円を、混合正規分布はなぞれます。k-meansは丸で切ります。4本の列で3つに分けると、種と食い違うのは5羽でした。ここまで、群の数3は、こちらで決めて渡してきました。


### お手本：4本を標準化

- 役割：4本の列を、平均0・標準偏差1にそろえる
- 入力：表`df`の4本の列`cols`
- 出力：配列`X`（羽の数×4）


In [ ]:
X = StandardScaler().fit_transform(df[cols])
print(X.shape)


#### 1行ずつ読む

- 1行目：4本の列をそろえて、配列`X`にする
- 2行目：形（羽の数×列の数）を見る


### 群の数を選ぶ物差し：BIC

混合正規分布は、データがどのくらい出てきやすいか（尤度）で、当てはまりを測れます。でも、成分を増やすと、尤度は上がり続けます。それでは数が決まりません。そこで、成分を増やした分のペナルティを付けて比べます。

$$
\mathrm{BIC} = -2\,\ell + d \log n
$$

$\ell$は対数尤度（当てはまりのよさ）、$d$はパラメータの数、$n$は羽の数です。第1項は当てはまりの悪さ、第2項は成分を増やした分のペナルティです。小さいほどよく、いちばん小さい群の数を選びます。


### お手本：群の数ごとのBIC

- 役割：成分の数を1から8まで替えて、BICをリストにためる
- 入力：配列`X`
- 出力：リスト`bic_full`（8つのBIC）


In [ ]:
bic_full = []
for k in range(1, 9):
    gm_k = GaussianMixture(k, n_init=5, random_state=0)
    gm_k.fit(X)
    bic_full.append(round(gm_k.bic(X)))
print(bic_full)


#### 1行ずつ読む

- 1行目：空のリストを用意する（第4回）
- 2〜4行目：成分の数`k`を1から8まで替えて、当てはめる
- 5行目：BIC（`bic`）を整数に丸めて、リストの後ろに足す
- 6行目：8つのBICを表示する


#### 中身を見る：3回だけ

くり返しを3回だけ回し、1回ごとにリストを表示します。名前は`bic_b`にして、上の`bic_full`を壊さないようにします。


In [ ]:
bic_b = []
for k in range(1, 4):
    gm_k = GaussianMixture(k, n_init=5, random_state=0)
    gm_k.fit(X)
    bic_b.append(round(gm_k.bic(X)))
    print(k, bic_b)


3がいちばん小さくなりました。種の数と同じです。


### 群の形には、型がある

| 型（`covariance_type`） | 群の形 |
|---|---|
| `full` | 群ごとに、広がりも傾きも違う楕円 |
| `tied` | どの群も、同じ形の楕円 |
| `diag` | 群ごとに違うが、軸に沿った楕円（傾かない） |
| `spherical` | 群ごとに大きさの違う丸 |

上のお手本は、何も書かないので`full`でした。


In [ ]:
# @title 図：型ごとに、くちばし2本を3つに分ける（中身は読み飛ばしてよい）
fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharey=True)
for ax, t in zip(axes, ['full', 'tied', 'diag', 'spherical']):
    g = GaussianMixture(3, covariance_type=t, n_init=5, random_state=0)
    g.fit(X2)
    ax.scatter(X2[:, 0], X2[:, 1], s=6, c=g.predict(X2))
    for j in range(3):
        if t == 'full':
            S = g.covariances_[j]
        elif t == 'tied':
            S = g.covariances_
        elif t == 'diag':
            S = np.diag(g.covariances_[j])
        else:
            S = np.eye(2) * g.covariances_[j]
        vals, vecs = np.linalg.eigh(S)
        ang = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
        ax.add_patch(Ellipse(g.means_[j], 4 * np.sqrt(vals[1]), 4 * np.sqrt(vals[0]),
                             angle=ang, fill=False, lw=2))
    ax.set_title(t)
    ax.set_aspect('equal')
plt.show()


くちばし2本で、型ごとに3つに分けました。型が狭いほど、斜めの塊をなぞれません。


### 型と数の、すべての組

型を替えても、BICで選ぶ数は3でしょうか。型4つ×群の数8つ、すべての組でBICを出したいところです。そこで、`for`の中に、もう1つ`for`を入れます。


### 入れ子のfor：最小の例

- 役割：2つのリストの、すべての組を表示する
- 入力：なし
- 出力：6つの組


In [ ]:
for t in ['丸', '楕円']:
    for k in [1, 2, 3]:
        print(t, k)


#### 1行ずつ読む

- 1行目：外側。`t`に「丸」「楕円」を順に入れる
- 2行目：内側。外側の1回ごとに、`k`に1・2・3を全部入れる
- 3行目：組を表示する。外側2回×内側3回で、6回

外側の1回ごとに、内側が全部回り、表が1行ずつ埋まります。


### お手本：型と数のBICの表

- 役割：型4つ×群の数1〜8の、すべての組でBICを出し、表にする
- 入力：配列`X`
- 出力：表`tab`（行が群の数、列が型）


In [ ]:
types = ['full', 'tied', 'diag', 'spherical']
bic = {}
for t in types:
    bic[t] = []
    for k in range(1, 9):
        gm_k = GaussianMixture(k, covariance_type=t, n_init=5,
                               random_state=0)
        gm_k.fit(X)
        bic[t].append(round(gm_k.bic(X)))
tab = pd.DataFrame(bic, index=range(1, 9))
tab


#### 1行ずつ読む

- 1・2行目：4つの型のリストと、結果をためる空の辞書（第5回）を作る
- 3行目：外側。型`t`を1つずつ取る
- 4行目：その型の鍵に、空のリストを入れる
- 5〜8行目：内側。群の数`k`を1から8まで替え、型`t`で当てはめる
- 9行目：BICを、その型のリストの後ろに足す
- 10・11行目：辞書を表にする。行が群の数、列が型


#### 中身を見る：2つの型、2つの数

型を2つ、群の数を2つにして、内側の1回ごとに辞書を表示します。名前は`bic_b`にして、上の`bic`を壊さないようにします。


In [ ]:
bic_b = {}
for t in ['full', 'spherical']:
    bic_b[t] = []
    for k in range(1, 3):
        gm_k = GaussianMixture(k, covariance_type=t, n_init=5,
                               random_state=0)
        gm_k.fit(X)
        bic_b[t].append(round(gm_k.bic(X)))
        print(t, k, bic_b)


外側の型が1つ進むごとに、内側の群の数が全部回ります。辞書の中のリストが、1つずつ伸びていきます。


> 今日の重要概念（Python）：入れ子のfor
>
> forの中にforを入れると、外側の1回ごとに、内側が全部回る。2つの選び方の、すべての組を試せる。


### お手本：型ごとに選ばれる数

- 役割：型ごとに、BICがいちばん小さい群の数を出す
- 入力：表`tab`
- 出力：型ごとの群の数


In [ ]:
print(tab.idxmin())


#### 1行ずつ読む

- 1行目：列（型）ごとに、BICがいちばん小さい行（群の数）を表示する（`idxmin`）


In [ ]:
for t in types:
    plt.plot(tab.index, tab[t], 'o-', label=t)
plt.ylim(2450, 3200)
plt.xlabel('群の数（成分の数）')
plt.ylabel('BIC（小さいほどよい）')
plt.legend()
plt.show()


えっ、楕円の型（`full`）は3ですが、丸の型（`spherical`）と、軸に沿った楕円（`diag`）は7を選びました。

丸の型では、種ごとの斜めの楕円を、1つの丸では覆えません。そこで、小さな丸をいくつも並べて、継ぎはぎします。BICが選ぶのは、その形のもとで、データにいちばん近い重ね方です。


> 今日の重要概念（統計）：成分は群とは限らない
>
> BICが選ぶのは、置いた形の正規分布を何個重ねるかで、群の数ではない。形の仮定を替えると、選ばれる数も変わる。


### 表全体でも、3ではない


In [ ]:
print(tab.min())


表全体でいちばん小さいのは、`tied`の4つ（2501）です。「BICで選べば種の数」とは言えません。


### TODO②：列を2本にする（予想 → 実行 → 比べる）

下のセルは、選んだ列で、型4つ×群の数1〜8のBICを出し、型ごとに選ばれる群の数を表示します。

まず、予想を書いてください。くちばし2本にすると、丸の型はまだ7つを選ぶでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`cols`を`cols2`に替えて実行してください。


In [ ]:
# TODO: cols を cols2 に替えて実行する

X_x = StandardScaler().fit_transform(df[cols2])
bic_x = {}
for t in types:
    bic_x[t] = []
    for k in range(1, 9):
        gm_k = GaussianMixture(k, covariance_type=t, n_init=5,
                               random_state=0)
        gm_k.fit(X_x)
        bic_x[t].append(round(gm_k.bic(X_x)))
print(pd.DataFrame(bic_x, index=range(1, 9)).idxmin())


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。丸の型が選ぶ数は、変わりましたか。


（ここに答えを書く）


くちばし2本では、楕円の型（`full`）が3、ほかの3つの型は4を選びます。使う列を替えても、選ばれる数は変わります。数は、列と形の仮定しだいです。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 楕円を許す（1節） | 種と食い違う羽：k-means 25、混合正規分布 12（くちばし2本） |
| 群の数を選ぶ（2節） | BICは、楕円の型なら3、丸の型なら7。形の仮定で数が変わる |
| 境目を読む（3節） | ？（このあとの段で） |
| 名前を付ける（4節） | ？（このあとの段で） |
| 種の中を分ける（5節） | ？（このあとの段で） |

まだ分からないのは、群に分けた1羽1羽を、言い切ってよいか、です。


---


## 3. 言い切れる？


ここから、段3「境目を読む」（3節）です。問いは、群に分けた1羽1羽を、その群だと言い切ってよいか、です。

BICで選ぶ数は、形の仮定しだいでした。楕円の型なら3、丸の型なら7です。楕円の型で3つに分けると、種とよく合います。ここから、楕円の型で3つに分けた結果を、1羽ずつ読みます。


### 確率で言える

第8回のk-meansは、どの星も1つの群に言い切りました。混合正規分布は、各成分から出てきたらしさを、確率で出せます。境目の羽は、どちらとも言い切れない、と言えます。

羽$i$が成分$j$から出てきた確率は、次の式です。

$$
P(j \mid x_i) = \frac{\pi_j \, N(x_i \mid \mu_j, \Sigma_j)}{\sum_{l=1}^{k} \pi_l \, N(x_i \mid \mu_l, \Sigma_l)}
$$

分子は、成分$j$の割合と出てきやすさ、分母は全部の成分の和です。1羽について、$k$個の確率を足すと1になります。


### お手本：所属の確率

- 役割：4本の列で楕円の型の3成分を当てはめ、羽ごとに3つの成分の確率を出す
- 入力：配列`X`
- 出力：配列`P`（羽の数×3）


In [ ]:
gm = GaussianMixture(3, n_init=5, random_state=0)
gm.fit(X)
P = gm.predict_proba(X)
print(P[:3].round(3))


#### 1行ずつ読む

- 1・2行目：4本の列で、楕円の型（full）の3成分を当てはめる
- 3行目：羽ごとに、3つの成分の確率を出す（`predict_proba`）。1行が1羽
- 4行目：先頭3羽の確率を見る


#### 中身を見る：1羽ぶん


In [ ]:
print(P[72].round(3), P[72].sum().round(3))


72番の羽は、2つの成分のあいだで迷っています。3つを足すと1です。


### お手本：言い切れない羽を数える

- 役割：いちばん大きい確率が0.9に届かない羽を数える
- 入力：配列`P`
- 出力：羽の数


In [ ]:
pmax = P.max(axis=1)
print((pmax < 0.9).sum(), '羽')


#### 1行ずつ読む

- 1行目：羽ごとに、3つの確率のうちいちばん大きいものを取る（`axis=1`は横に）
- 2行目：それが0.9に届かない羽を数える（Trueを1として足す）


### お手本：言い切れない羽を見る


In [ ]:
df[pmax < 0.9][['種', '性別', 'くちばしの長さ', 'くちばしの深さ']]


条件に合う行だけを取り出し、4本の列を見ました。


### お手本：境目に印を付ける

- 役割：全部の羽を成分の色で打ち、言い切れない羽に×を重ねる
- 入力：表`df`、`gm`、`pmax`
- 出力：図


In [ ]:
plt.scatter(df['くちばしの長さ'], df['くちばしの深さ'], s=10,
            c=gm.predict(X))
amb = df[pmax < 0.9]
plt.scatter(amb['くちばしの長さ'], amb['くちばしの深さ'], s=80,
            marker='x', color='black')
plt.xlabel('くちばしの長さ（mm）')
plt.ylabel('くちばしの深さ（mm）')
plt.show()


#### 1行ずつ読む

- 1・2行目：全部の羽を、いちばん出てきたらしい成分の色で打つ
- 3行目：言い切れない羽だけの表`amb`を作る
- 4・5行目：その羽に、黒い×を大きく重ねる
- 6〜8行目：軸の名前を付けて、図を表示する

×は、アデリーとヒゲの塊のあいだにいます。7羽のうち5羽は、ヒゲのメスです。小さめのヒゲは、アデリーに似ます。k-meansでは、この羽も、どれか1つの群に言い切られていました。


> 重要：混合正規分布は、どの群に入るかを確率で言える。言い切れない点を、言い切れないまま残せる。


### TODO③：しきい値を替える（予想 → 実行 → 比べる）

下のセルは、いちばん大きい確率が0.9に届かない羽を数えます。

まず、予想を書いてください。0.6に届かない羽は、何羽になるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`0.9`を`0.6`に替えて実行してください。


In [ ]:
# TODO: 0.9 を 0.6 に替えて実行する

print((pmax < 0.6).sum(), '羽')


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。何羽になりましたか。


（ここに答えを書く）


0.6に届かない羽は2羽です。ほぼ半々で迷っている羽は、わずかです。しきい値は、こちらが決めます。決めた値をレポートに書きます。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 楕円を許す（1節） | 種と食い違う羽：k-means 25、混合正規分布 12（くちばし2本） |
| 群の数を選ぶ（2節） | BICは、楕円の型なら3、丸の型なら7。形の仮定で数が変わる |
| 境目を読む（3節） | 所属の確率が0.9に届かない羽が7羽。種の境目にいる |
| 名前を付ける（4節） | ？（このあとの段で） |
| 種の中を分ける（5節） | ？（このあとの段で） |

まだ分からないのは、丸の型が選んだ7つの成分は、何の群か、です。


---


## 4. 7つの成分は何？


ここから、段4「名前を付ける」（4節）です。問いは、丸の型でBICが選んだ7つの成分は、何の群か、です。

混合正規分布は、所属を確率で言えます。言い切れない羽は7羽でした。丸の型のBICは7つを選びました。種は3つなのに。ここから、7つの成分に名前を付け、使わなかった列で確かめます。


### お手本：丸の型で7つに

- 役割：丸の型で、成分の数7で当てはめ、羽ごとの成分の番号を列にする
- 入力：配列`X`
- 出力：列「成分」と、成分ごとの羽の数


In [ ]:
gm7 = GaussianMixture(7, covariance_type='spherical', n_init=5,
                      random_state=0)
gm7.fit(X)
df['成分'] = gm7.predict(X)
print(df['成分'].value_counts().sort_index().tolist())


#### 1行ずつ読む

- 1〜3行目：丸の型で、成分の数7で当てはめる
- 4行目：羽ごとの成分の番号を、列「成分」にする
- 5行目：成分0から6の順に、羽の数を表示する


### お手本：成分ごとの平均

- 役割：成分ごとに、4本の列の平均を出す
- 入力：表`df`
- 出力：成分×4本の表


In [ ]:
df.groupby('成分')[cols].mean().round(0)


#### 1行ずつ読む

- 1行目：成分ごとに羽を分け（`groupby`）、4本の列の平均を出す

ひれが長く重い成分が3つ、くちばしが短い成分が2つ、くちばしが長く深い成分が2つあります。ひれが長く重い3つはジェンツーらしく、くちばしが短い2つはアデリー、長く深い2つはヒゲらしいと読めます。同じ種らしい成分のあいだでは、体重が違います。


### お手本：種と突き合わせる


In [ ]:
pd.crosstab(df['種'], df['成分'])


どの成分も、ほぼ1つの種だけです。名前の候補は合っていました。ただし、1つの種が2つか3つの成分に分かれました。では、種の中の何で分かれたのでしょうか。


### お手本：島と突き合わせる


In [ ]:
pd.crosstab(df['島'], df['成分'])


アデリーらしい成分は、3つの島すべてにまたがります。島では分かれていません。


> 重要：成分ごとの平均から付けた名前は、候補にすぎない。使わなかった列と突き合わせて、確かめてから名前にする。


### TODO④：突き合わせる列を替える（予想 → 実行 → 比べる）

下のセルは、島と成分の組ごとに羽を数えます。

まず、予想を書いてください。性別と突き合わせると、同じ種の成分どうしは、オスとメスに分かれているでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`'島'`を`'性別'`に替えて実行してください。


In [ ]:
# TODO: '島' を '性別' に替えて実行する

pd.crosstab(df['性別'], df['成分'])


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。同じ種の成分どうしは、オスとメスに分かれていましたか。


（ここに答えを書く）


同じ種の成分どうしは、オスの成分とメスの成分に分かれていました（性別が欠測の9羽は数に入りません）。オスはメスより大きいので、丸の型は、種の中のオスとメスの違いまで、別の成分にしたのです。


In [ ]:
# @title 図：丸の型の7つの成分と、種と性別（中身は読み飛ばしてよい）
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
axes[0].scatter(df['ひれの長さ'], df['体重'], s=12, c=df['成分'], cmap='tab10')
axes[0].set_title('丸の型の7つの成分')
for s in ['アデリー', 'ヒゲ', 'ジェンツー']:
    for x in ['オス', 'メス']:
        d = df[(df['種'] == s) & (df['性別'] == x)]
        axes[1].scatter(d['ひれの長さ'], d['体重'], s=12, label=f'{s}・{x}')
axes[1].set_title('種と性別')
axes[1].legend(fontsize=9)
for ax in axes:
    ax.set_xlabel('ひれの長さ（mm）')
axes[0].set_ylabel('体重（g）')
plt.show()


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 楕円を許す（1節） | 種と食い違う羽：k-means 25、混合正規分布 12（くちばし2本） |
| 群の数を選ぶ（2節） | BICは、楕円の型なら3、丸の型なら7。形の仮定で数が変わる |
| 境目を読む（3節） | 所属の確率が0.9に届かない羽が7羽。種の境目にいる |
| 名前を付ける（4節） | 丸の7つの成分は、種と性別の組にかなり重なる |
| 種の中を分ける（5節） | ？（このあとの段で） |

まだ分からないのは、楕円の型でも、種の中にオスとメスの群が見つかるか、です。


---


## 5. 種の中にも群はある？


ここから、段5「種の中を分ける」（5節）です。問いは、楕円の型で1つの種だけを分けても、オスとメスの群は見つかるか、です。

丸の型の7つの成分は、種とオス・メスの組にかなり重なりました。丸の型は、斜めの楕円を継ぎはぎするので、余った成分ができやすいのでした。楕円を許す型でも、種の中に群はあるのでしょうか。

1つの種の羽だけを取り出し、4本の列を、その種の中で標準化します。楕円の型（full）で、成分の数2で分けます。性別は使わず、分けたあとの答え合わせだけに使います。


### お手本：ジェンツーだけを2つに

- 役割：ジェンツーの羽だけで混合正規分布を当てはめ、性別と成分の組を数える
- 入力：表`df`と4本の列`cols`
- 出力：性別×成分の表


In [ ]:
gen = df[df['種'] == 'ジェンツー']
Xg = StandardScaler().fit_transform(gen[cols])
gm_g = GaussianMixture(2, n_init=5, random_state=0)
gm_g.fit(Xg)
pd.crosstab(gen['性別'], gm_g.predict(Xg))


#### 1行ずつ読む

- 1・2行目：ジェンツーの羽だけを取り出し、4本の列を標準化する
- 3・4行目：楕円の型で、成分の数2で当てはめる
- 5行目：性別と成分の組ごとに数える


#### 中身を見る：成分ごとの平均


In [ ]:
gen.groupby(gm_g.predict(Xg))[cols].mean().round(0)


成分0は、くちばしが長く深く、ひれが長く、重い成分です。オスらしい成分です。


In [ ]:
# @title 図：ジェンツーの成分（色）と性別（印）（中身は読み飛ばしてよい）
lab_g = gm_g.predict(Xg)
for x, mk in [('オス', '^'), ('メス', 'o')]:
    for j in range(2):
        m = (gen['性別'] == x).values & (lab_g == j)
        plt.scatter(gen['ひれの長さ'][m], gen['体重'][m], marker=mk,
                    label='成分' + str(j) + '・' + x)
plt.legend()
plt.xlabel('ひれの長さ（mm）')
plt.ylabel('体重（g）')
plt.show()


メス（丸）は、すべて成分1に入りました。オス（三角）は多くが成分0で、小さめのオスが成分1にまざります。


> 重要：種という群の中にも、オスとメスという群がある。データはどちらにも分けられるので、何を群と呼ぶかは、問いが決める。


### TODO⑤：種を替える（予想 → 実行 → 比べる）

下のセルは、1つの種の羽だけで混合正規分布を当てはめ、性別と成分の組を数えます。

まず、予想を書いてください。アデリーでも、メスは1つの成分にまとまるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`'ジェンツー'`を`'アデリー'`に替えて実行してください。


In [ ]:
# TODO: 'ジェンツー' を 'アデリー' に替えて実行する

sp = df[df['種'] == 'アデリー']
Xs = StandardScaler().fit_transform(sp[cols])
gm_s = GaussianMixture(2, n_init=5, random_state=0)
gm_s.fit(Xs)
pd.crosstab(sp['性別'], gm_s.predict(Xs))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。メスは、1つの成分にまとまりましたか。


（ここに答えを書く）


アデリーでは、メス73羽のうち72羽が成分0、オス73羽のうち52羽が成分1に入ります。ジェンツーと同じく、オスとメスにかなり分かれます。小さめのオスが、メスの成分にまざります。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 楕円を許す（1節） | 種と食い違う羽：k-means 25、混合正規分布 12（くちばし2本） |
| 群の数を選ぶ（2節） | BICは、楕円の型なら3、丸の型なら7。形の仮定で数が変わる |
| 境目を読む（3節） | 所属の確率が0.9に届かない羽が7羽。種の境目にいる |
| 名前を付ける（4節） | 丸の7つの成分は、種と性別の組にかなり重なる |
| 種の中を分ける（5節） | ジェンツーを2つに分けると、メス58羽はすべて同じ成分 |

5つの段がそろい、今日の問いに答えられます。


### 答え

楕円を許す混合正規分布なら、体の測り方だけで3種はほぼ分かれます（食い違い5羽）。BICが選ぶ数は、型と列しだいで、3にも4にも7にもなります。丸の型の7つは、種とオス・メスの組でした。余った成分にも、意味がありました。


まだ言えないこと

- どの型が正しいかは、データだけでは決まらない
- 種を群と呼ぶか、種とオス・メスの組を群と呼ぶかは、問いが決める
- 性別が欠測の9羽が、どちらの性別か


### レポートなら、こう書く

今日の解析を、期末レポートの項目に写すと、次のようになります。

- 予想：体の測り方だけで、3種に分かれるだろう
- 解析：4本の列を標準化し、混合正規分布で3つに分けた。種と食い違うのは5羽
- 群の数：型4つ×群の数1〜8のBICを比べた。楕円の型は3、丸の型は7
- 驚き：丸の型の7つの成分は、種とオス・メスの組だった
- 境目：所属の確率が0.9に届かない羽が7羽。多くはヒゲのメス
- 限界：BICの数は、型と列の選び方しだい。何を群と呼ぶかは、問いで決めた


---


## 目標に答えられたか


- 今日の目標は「体の測り方だけで、ペンギンの種は分かれるか。何種類とみなせるか」でした。図と、食い違いの数と、BICの表で答えられますか？
- 1節で、k-meansが斜めの塊の端を、となりの群に取られたのはなぜですか？
- 2節で、丸の型のBICが7つを選んだのはなぜですか？
- 2節の入れ子の`for`で、内側の`for`は全部で何回回りましたか？
- 3節で、所属の確率は、k-meansの群の番号と比べて、何が言えるようになりましたか？
- 4節で、成分の名前を確かめるのに、なぜ使わなかった列を使ったのですか？


---


## 今日の要点


- 今日、持ち帰る1文：混合正規分布は、群ごとに楕円を許し、所属を確率で言う。BICが選ぶのは置いた形の成分の数で、群の数ではない。名前は、使わなかった列で確かめて付ける
- 重要概念（統計）：成分は群とは限らない。BICが選ぶのは、置いた形の正規分布を何個重ねるかで、群の数ではない。形の仮定を替えると、選ばれる数も変わる
- 重要概念（Python）：入れ子のfor。forの中にforを入れると、外側の1回ごとに、内側が全部回る。2つの選び方の、すべての組を試せる
- k-meansは丸い群、混合正規分布は楕円の群を探す
- 群の名前は候補。使わなかった列と突き合わせて確かめる


---


## 次回へ


ここまで、自分たちでデータを選び、解析の手順を組み立ててきました。では、研究者が論文に載せた図は、どんなコードから生まれたのでしょうか。


---


## 課題（提出）


提出するもの：絵2枚（画像ファイル）。今日の重要概念の、統計とPythonを1枚ずつです。提出フォームから出してください。期限はありません。

コードは書きません。上手な絵である必要はありません。紙に手で描いて写真を撮っても、描画アプリで描いてもかまいません。


### 今日の重要概念（2つ）

- 統計：成分は群とは限らない。BICが選ぶのは、置いた形の正規分布を何個重ねるかで、群の数ではない。形の仮定を替えると、選ばれる数も変わる（2節）
- Python：入れ子のfor。forの中にforを入れると、外側の1回ごとに、内側が全部回る。2つの選び方の、すべての組を試せる（2節）


### 今日の2枚（絵にする）

2つの重要概念を、1枚ずつ絵にしてください。その考えが1枚で伝わる絵です。数式やコードは描かなくてかまいません。半年後の自分が見て、何の話だったか思い出せる絵を目指します。


<details><summary>詰まったら</summary>

成分は群とは限らない、なら、たとえば「丸いマットで床を覆う」を描けます。
細長い廊下と丸い部屋の床に、丸いマットだけを敷いていきます。丸い部屋は1枚で足ります。細長い廊下は、何枚も並べないと覆えません。マットの枚数は、部屋の数ではありません。マットの形しだいで、枚数が変わります。2節の、型ごとに3つに分けた図が手本になります。

</details>


<details><summary>詰まったら</summary>

入れ子のforなら、たとえば「九九の表を埋める」を描けます。
1の段（外側の1回）のあいだに、1から9（内側）を全部かけます。段が1つ進むごとに、表が1行ずつ埋まります。2節の、型と群の数のBICの表が手本になります。

</details>


---


## 発展（任意）


### 性別が欠測の羽を推す

アデリーには、性別が欠測の羽が5羽います。5節で、種の中で2つに分けた成分は、オスとメスにかなり重なりました。欠測の羽が、どちらの成分にどれだけの確率で入るかを見ます。


In [ ]:
ade = df[df['種'] == 'アデリー']
Xa = StandardScaler().fit_transform(ade[cols])
gm_a = GaussianMixture(2, n_init=5, random_state=0)
gm_a.fit(Xa)
na = ade['性別'].isna().values
print(gm_a.predict_proba(Xa)[na].round(2))


#### 1行ずつ読む

- 1〜4行目：アデリーだけで、成分の数2の混合正規分布を当てはめる（TODO⑤と同じ）
- 5行目：性別が欠測ならTrueの配列
- 6行目：欠測の羽の、2つの成分の所属の確率

TODO⑤で、成分0はメス72羽と小さめのオス21羽、成分1はオス52羽とメス1羽でした。5羽のうち4羽は成分0（メスらしい）、1羽は成分1（オスらしい）に、確率0.97以上で入ります。ただし、成分0には小さめのオスもまざるので、「メスらしい」とは言えても、メスと決めつけはできません。

試すなら、`'アデリー'`を`'ジェンツー'`に替えてみましょう。欠測の4羽は、どちらの成分に入るでしょうか。
